# The mobile base

Reachy 2 is mounted on a mobile base!

## Initialize your robot

First connect to your robot:

In [1]:
from reachy2_sdk import ReachySDK
import time

reachy = ReachySDK(host='10.0.0.201')  # Replace with the actual IP

Let's check what contains the mobile base part:

In [2]:
reachy.mobile_base

<MobileBase on=False 
 lidar_safety_enabled=True 
 battery_voltage=25.7>

## Move around

Three modes are possible to control the mobile base:
- **goto** : move the mobile base to a target point in space -> use a *goto function* to get in this mode
- **free wheel**: unlock the wheel so Reachy can be manually moved around easily -> *turn_off() method* will set this mode
- **brake**: stop the movement and lock the wheels -> *turn_on() method* will set this mode

The goto commands and based commands described below follow all the rules you saw in the [goto introduction tutorial](2_goto_introduction.ipynb).

### Goto and odometry

The goto function is used to place the mobile_base at a relative position and orientation to its odometry, set when the robot is switched on. To be sure, you can reset the odometry before calling the function. 

In [3]:
reachy.mobile_base.reset_odometry()

In [5]:
reachy.mobile_base.odometry

{'x': 0.0, 'y': 0.0, 'theta': 0.0, 'vx': 0.0, 'vy': 0.0, 'vtheta': 0.0}

The robot is now positionned at x=0, y=0, theta=0.  
If you want to move forward again the robot, you need to increase the x value (value is in meters):

In [16]:
# Move 20 cm forward
a = reachy.mobile_base.goto(x=0.2, y=0.0, theta=0.0)

distance_tolerance 0.05


Now, request `goto(0, 0, 0)`. The robot will return to its previous position:

In [50]:
reachy.mobile_base.goto(x=0.0, y=0.0, theta=0.0)

id: 45

All the positions are relative to the fixed coordinate system of the mobile base set when the odometry is set, which means at the start of the robot of after a `reset_odometry()` asked by the user.

So if you do:

In [ ]:
# Move 30cm forward, to reach x=30cm
reachy.mobile_base.goto(x=0.3, y=0.0, theta=0.0)

# Go back by 10cm, to reach x=20cm
reachy.mobile_base.goto(x=0.2, y=0.0, theta=0.0)

The mobile is first going to the position x=30cm in the odometry frame. We then ask for it to get to the position x=20cm in this same frame, so the mobile base is going backward by 10cm to reach its new target.  

Let's do the same by resetting the odometry between the two commands:

In [ ]:
# Move 30cm forward, to reach x=30cm in the current odometry frame
reachy.mobile_base.goto(x=0.3, y=0.0, theta=0.0, wait=True)
print(f"x position before odometry reset: {reachy.mobile_base.odometry['x']}")

# Reset odometry
reachy.mobile_base.reset_odometry()
print(f"x position after odometry reset: {reachy.mobile_base.odometry['x']}")

# Move 20cm forward, to reach x=20cm in the new current odometry frame
reachy.mobile_base.goto(x=0.2, y=0.0, theta=0.0)

As we reset the odometry between the two commands, the mobile base odometry position is reset to x=0cm before the second command. It will then reach x=20cm in the new frame, so move forward by 20cm.

We recommend experimenting with this concept to get familiar.

### Goto tolerances and timeout

### Relative moves

Two methods are available to give relative orders:
- **`translate_by()`**: to give translations orders. 
- **`rotate_by()`**: to give rotations orders  

The translation or rotation is computed based on the current position if no goto is playing, or on the position required for the last queued or playing goto in case gotos are not finished.

In [6]:
a=reachy.mobile_base.goto(x=0.2, y=0.0, theta=0, timeout=20)

In [8]:
reachy.mobile_base._get_goto_request(a)

SimplifiedRequest(part='mobile_base', request=OdometryRequest(goal_positions={'x': 0.20000000298023224, 'y': 0.0, 'theta': 0.0}, timeout=20.0, distance_tolerance=0.05000000074505806, angle_tolerance=0.0872664600610733))

If you want to move forward again the robot, you need to increase the x value.

In [10]:
import time

reachy.mobile_base.goto(x=0.2, y=0.0, theta=0.0) #that won't do anything as the robot is already there
time.sleep(2)

# Move again 20cm forward
reachy.mobile_base.goto(x=0.4, y=0.0, theta=0.0)

id: 14

That's the same for the rotation. You can go back to the initial position than rotate the mobile base. 

In [9]:
# Go back to the initial position
reachy.mobile_base.goto(x=0.0, y=0.0, theta=0.0)

time.sleep(2)

# Rotation to be at 90 degrees in the frame
reachy.mobile_base.goto(x=0.0, y=0.0, theta=90.0, wait=True)

distance_tolerance 0.05
distance_tolerance 0.05


id: 30

In [11]:
reachy.mobile_base.goto(x=0.2, y=0.0, theta=0, wait=True, distance_tolerance=0.1, timeout=5)
reachy.mobile_base.goto(x=0.1, y=0.0, theta=0, wait=True, distance_tolerance=0.02, timeout=5)
reachy.mobile_base.goto(x=0.0, y=0.0, theta=0, wait=True, distance_tolerance=0.02)

distance_tolerance 0.1
distance_tolerance 0.02
distance_tolerance 0.02


id: 37

In [12]:
reachy.mobile_base.goto(x=0.2, y=0.0, theta=90, wait=True, angle_tolerance=10)
reachy.mobile_base.goto(x=0.2, y=0.0, theta=20, wait=True, angle_tolerance=1, timeout=5)

distance_tolerance 0.05
distance_tolerance 0.05


id: 39

# Else, not for you Remi

In [18]:
# Go back to 0 degree in the frame
reachy.mobile_base.goto(x=0.0, y=0.0, theta=0.0, wait=True)

distance_tolerance 0.05


id: 44

In [ ]:
# Rotation to be at 90 degrees in the frame
reachy.mobile_base.goto(x=0.0, y=0.0, theta=90.0, wait=True)

# Reset odometry
reachy.mobile_base.reset_odometry()
# Go back to 0 degree in the frame : it won't move because the frame has changed
reachy.mobile_base.goto(x=0.0, y=0.0, theta=0.0, wait=True)

In [ ]:
# Rotation to be at -90 degrees in the new frame
reachy.mobile_base.goto(x=0.0, y=0.0, theta=-90.0)

#### Rotate_by / Translate_by

You can also decide to assign movements to the robot based on its current position and not on its odometry. 

In [14]:
reachy.mobile_base.get_current_odometry()
reachy.mobile_base.translate_by(x = 0.2, y = 0.0)

distance_tolerance 0.05


id: 41

You can do it again, to move the robot by 0.2m in the x direction.

In [15]:
a=reachy.mobile_base.translate_by(x = 0.2, y = 0.0)

distance_tolerance 0.05


In [16]:
reachy.get_goto_request(a)

SimplifiedRequest(part='mobile_base', request=OdometryRequest(goal_positions={'x': 0.40611156821250916, 'y': -0.030791906639933586, 'theta': -0.056279897689819336}, timeout=100.0, distance_tolerance=0.05000000074505806, angle_tolerance=0.0872664600610733))

Now, the mobile base can be rotated 90° from its current position, allowing to get a odometry with a theta = 0°. 

In [17]:
reachy.mobile_base.rotate_by(theta = 90.0)
reachy.mobile_base.get_current_odometry()

distance_tolerance 0.05


{'x': 0.41116631031036377,
 'y': -0.04451792687177658,
 'theta': -1.0564855499679329,
 'vx': 0.0,
 'vy': 0.0,
 'vtheta': 0.0}

The speed of the movement can be defined using this command : *this will assign speed to the robot for 200ms*

In [20]:
reachy.mobile_base.set_goal_speed(x=1.0, y=0.0, theta=0)
tic=time.time()
while time.time()-tic < 5:
    reachy.mobile_base.send_speed_command()
    time.sleep(0.01)

### Free wheel

In [21]:
reachy.mobile_base.turn_off()